# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR\u02c62 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and associated fields. All Croissant entities are referenced by their `@id` for consistency.

In [ ]:
# List all record sets available in the Croissant metadata

# Get all record sets from the metadata
record_sets = getattr(metadata, 'record_sets', None)

if not record_sets:
    # Alternative: scan distributions as RecordSets may be encoded there
    print("No explicit record sets found; using distributions as record sets.")
    record_sets = [d['@id'] for d in (getattr(metadata, 'distribution', []) if hasattr(metadata, 'distribution') else [])]
else:
    # RecordSets present: get their @ids
    record_sets = [rs['@id'] for rs in record_sets] if isinstance(record_sets[0], dict) else record_sets

if not record_sets:
    raise RuntimeError("Could not find any record sets or distributions in this Croissant metadata.")

# Print available record sets
print("Available record sets (by @id):")
for rs_id in record_sets:
    print(f"  - {rs_id}")

# For the main record set (assume first), list out field @ids
main_record_set = record_sets[0]

try:
    record_set_obj = None
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        for rs in metadata.record_sets:
            if rs.get('@id', None) == main_record_set:
                record_set_obj = rs
                break
    # Fallback: use distributions
    if record_set_obj is None and hasattr(metadata, 'distribution'):
        for distr in metadata.distribution:
            if distr.get('@id', None) == main_record_set:
                record_set_obj = distr
                break
    # Show fields/columns for this record set
    print(f"\nFields in the main record set (@id: {main_record_set}):")
    fields = None
    if record_set_obj and 'field' in record_set_obj:
        fields = record_set_obj['field']
    elif record_set_obj and 'column' in record_set_obj:
        fields = record_set_obj['column']
    if fields:
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                print(f" - {field['@id']}")
            else:
                print(f" - {field}")
    else:
        print("(Unable to list fields for this record set)")
except Exception as e:
    print("Could not extract fields:", str(e))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as identified above.

In [ ]:
# Extract data from each record set into pandas DataFrames using its @id
# Replace <record_set_id> with the main record set/distribution @id used above
dataframes = {}

for record_set in record_sets:
    try:
        # This references the record set by @id, as required
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {record_set}")
        print(f"Number of records: {len(df)}")
        print(f"Field IDs (columns): {list(df.columns)}\n")
        # Preview first 3 rows
        display(df.head(3))
        dataframes[record_set] = df
    except Exception as e:
        print(f"Could not load DataFrame for record set {record_set}: {str(e)}")

# For further analysis, we'll use the first DataFrame (main data table)
main_df_id = record_sets[0]
main_df = dataframes[main_df_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# EDA Example: Select a numeric field (by Croissant column @id/name) and group field
# Please update `numeric_field` and `group_field` to the actual field names or @ids from your record set
# We'll attempt to auto-select from typical protein datasets (e.g. 'MolecularWeight', 'Abundance', etc.), otherwise print available columns.

possible_numeric_fields = [
    'MW',            # Molecular weight
    'molecular_weight',
    'abundance',
    'Abundance',
    'normalized_abundance',
    'PeptideCount',
    'peptide_count',
    'Coverage',      # %
    'coverage',
    'pI',
    'score'
]

numeric_field = None
for f in main_df.columns:
    if f in possible_numeric_fields:
        numeric_field = f
        break
# If no common field found, ask user
if not numeric_field:
    print("No common numeric field found. Available columns:")
    print(list(main_df.columns))
    # For demonstration, pick the first numeric-looking column
    for f in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[f]):
            numeric_field = f
            break
    if not numeric_field:
        raise ValueError("No numeric field found for EDA.")

print(f"Using '{numeric_field}' as the numeric field for EDA.")

# Group by candidate field: look for 'Protein', 'accession', 'Group', etc.
possible_group_fields = ['accession', 'protein_group', 'Protein', 'Sample', 'modification', 'class']
group_field = None
for f in possible_group_fields:
    if f in main_df.columns:
        group_field = f
        break

if group_field:
    print(f"Using '{group_field}' as group field.")
else:
    print("No suitable group field found. Skipping grouping.")

# Filter and normalize
threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 10
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} ({len(filtered_df)} records)")

# Normalize column
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() or 1)

print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"Grouped data by {group_field}, showing mean {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution of the numeric field
plt.figure(figsize=(8, 5))
main_df[numeric_field].hist(bins=30)
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If grouping is possible, plot mean values per group
if group_field:
    plt.figure(figsize=(10, 5))
    top_groups = grouped_df.head(10)
    plt.bar(top_groups[group_field], top_groups[numeric_field])
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Top 10 {group_field}s by mean {numeric_field}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 protein mass spectrometry dataset via its Croissant schema using `mlcroissant`, reviewed the available data structures, extracted tables using only the official `@id` references, and performed basic analysis and visualizations. This approach demonstrates how to programmatically handle datasets following FAIR principles using the Croissant format and `mlcroissant` in Python.